In [ ]:
from turbofan_fc_cnn_mc import fc_cnn_mc
from util.postProcess import post
from util import prelim
import numpy as np
from saveasmat_mc import saveasmat_mc

In [ ]:
database = '../database2/BU_supaero_10_64_66_0.07m_0.16m_conv.json'

In [ ]:
# Load 2D data
rotor_test = [19, 24, 46] # Geometries to be used for testing.
rotor_drop = [] # Geometries to be removed from dataset.
input_keys = ["M_tip", "phi","th_le","th_te","x_le","x_te","radius_le","radius_te","dist_le","dist_te",
              "chord","stagger","metal_angle_le","metal_angle_te","thickness_upper_1","thickness_upper_2","thickness_upper_3",
             "thickness_upper_4","thickness_upper_5","thickness_upper_6","thickness_lower_1","thickness_lower_2",
             "thickness_lower_3","thickness_lower_4","thickness_lower_5","thickness_lower_6"] 
output_keys = ["tke", "LenScale", "vx", "vt", "p", "T", "v_mag"]
COORD_KEYS = ["x", "y", "z", "r", "th"]
input_, output_, coordinate, label = prelim.load_data_multiChan(
    file=database,
    input_keys=input_keys,
    output_keys=output_keys,total=True,metric=True)
allSlice = prelim.find_slice_by_geom_and_slice(label,geom_values=None, rpm_values = None, slice_values=[7])
evalSlice = prelim.find_slice_by_geom_and_slice(label,geom_values=rotor_test, rpm_values = [7000,9000,12000], slice_values=[7])
dropSlice = prelim.find_slice_by_geom_and_slice(label,geom_values=rotor_test, rpm_values = [15000], slice_values=[7])

In [ ]:
allSlice = range(input_.shape[0])
trainSlice = [i for i in allSlice if i not in set(evalSlice + dropSlice)]

In [ ]:
len(dropSlice)

In [ ]:
## Train/test spliting
reg_input_train, reg_input_test, reg_output_train, reg_output_test = prelim.regularized_train_test_mc(input_, output_, evalSlice, trainSlice=trainSlice, norm_path="normalize_parameters.pkl")
label_train, coordinate_train, label_test, coordinate_test = prelim.split_coordinate_label(coordinate, label, evalSlice, trainSlice=trainSlice)

In [ ]:
## Training multi-output model
mc_model = fc_cnn_mc(reg_input_train,reg_output_train)
#mc_model = fc_cnn_mc(reg_input_train,reg_output_train,reg_input_test,reg_output_test)
#mc_model.build_model(print_summary = True,lr=[0.002,1000,0.985])
mc_model.build_model(print_summary = True,lr=[0.0012,1000,0.99])
mc_model.fit_model(ep=1,p=100,ts=0.1,bs=128)
mc_model.model.save('Models/test_tke.h5')
#saveasmat_mc("Results/", mc_model.model,reg_input_train,reg_input_test,reg_output_train,reg_output_test,label_train,coordinate_train,label_test,coordinate_test,nfile='normalize_parameters.pkl')

In [ ]:
saveasmat_mc("Results/", mc_model.model,reg_input_train,reg_input_test,reg_output_train,reg_output_test,label_train,coordinate_train,label_test,coordinate_test,nfile='normalize_parameters.pkl')

In [ ]:
mc_model.plot_loss()
post(mc_model.model,reg_input_test,reg_output_test).pixelError_mc()
post(mc_model.model,reg_input_test,reg_output_test).circError_mc()

In [ ]:
mc_model = fc_cnn_mc(reg_input_train,reg_output_train)
mc_model.build_model(print_summary = True,lr=[0.001,1000,0.985])
from tensorflow.keras.models import load_model
mc_model.model = load_model('Models/model_mc.h5')